# KishoLens ML & NLP Feature Extraction Prototype

This notebook prototypes the core NLP processing and feature extraction modules of KishoLens. It loads crawled chapter text from SQLite (`data/kisholens.db`), checks for advanced NLP libraries like `spacy`, `sudachipy`, `nltk`, and `hanlp`, and computes detailed stylistic pacing features (dependency tree depth, parts of speech distribution, pronoun/adjective ratios, vocabulary diversity, and pacing metrics) for English, Japanese, and Chinese texts. If advanced libraries are not available (e.g. on Python 3.14 due to dependency availability), the pipeline automatically falls back to regex-based baselines.

In [1]:
import os
import re
from typing import Optional, List, Dict, Any
from sqlmodel import SQLModel, Field, Session, create_engine, select
import pandas as pd

# Try importing NLP packages for high-fidelity extraction
try:
    import spacy
    HAS_SPACY = True
except ImportError:
    HAS_SPACY = False

try:
    import sudachipy
    HAS_SUDACHI = True
except ImportError:
    HAS_SUDACHI = False

try:
    import nltk
    HAS_NLTK = True
except ImportError:
    HAS_NLTK = False

try:
    import hanlp
    HAS_HANLP = False  # Disabled by default in prototype to prevent memory/JIT locks
except ImportError:
    HAS_HANLP = False

print(f"Libraries available: spaCy={HAS_SPACY}, SudachiPy={HAS_SUDACHI}, NLTK={HAS_NLTK}, HanLP={HAS_HANLP}")

C:\Users\kingj\Documents\KishoLens\.venv\Lib\site-packages\torch\cuda\__init__.py:64: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Libraries available: spaCy=True, SudachiPy=True, NLTK=True, HanLP=False


In [2]:
# Initialize NLP models and fallback flags dynamically

nlp_en = None
nlp_ja = None
nlp_zh = None
nlp_hanlp = None

def load_spacy_model(model_name: str):
    """Helper to dynamically load or download a spaCy model, removing loading redundancies."""
    try:
        return spacy.load(model_name)
    except Exception as e:
        print(f"Could not load spaCy model {model_name}: {e}")
        return None

if HAS_SPACY:
    if spacy.util.is_package("en_core_web_sm"):
        nlp_en = load_spacy_model("en_core_web_sm")
    if HAS_SUDACHI and spacy.util.is_package("ja_core_news_sm"):
        nlp_ja = load_spacy_model("ja_core_news_sm")
    if spacy.util.is_package("zh_core_web_sm"):
        nlp_zh = load_spacy_model("zh_core_web_sm")

if HAS_NLTK:
    try:
        nltk.download('punkt', quiet=True)
        nltk.download('punkt_tab', quiet=True)
        nltk.download('averaged_perceptron_tagger', quiet=True)
        nltk.download('averaged_perceptron_tagger_eng', quiet=True)
        nltk.download('vader_lexicon', quiet=True)
    except Exception as e:
        print(f"Could not download NLTK resources: {e}")

if HAS_HANLP:
    try:
        nlp_hanlp = hanlp.load(hanlp.pretrained.mtl.CLOSE_TOK_POS_NER_SRL_DEP_SDP_CON_ELECTRA_SMALL_ZH)
        print("Loaded HanLP Chinese Pipeline")
    except Exception as e:
        print(f"Could not initialize HanLP model: {e}")


In [3]:
# Declare SQLModel tables

class Novel(SQLModel, table=True):
    __table_args__ = {"extend_existing": True}
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str
    author: str
    source: str

class Chapter(SQLModel, table=True):
    __table_args__ = {"extend_existing": True}
    id: Optional[int] = Field(default=None, primary_key=True)
    novel_id: int = Field(foreign_key="novel.id")
    chapter_number: int
    title: str
    text_ja: str
    text_en: str
    text_zh: str = Field(default="")

In [4]:
def compute_type_token_ratio(tokens: List[str]) -> float:
    """Computes Type-Token Ratio (vocabulary diversity)."""
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)


def compute_dep_tree_depth(doc) -> float:
    """
    Computes the average maximum dependency tree depth across all sentences in a spaCy document.
    """
    def get_depth(token):
        if not list(token.children):
            return 1
        return 1 + max(get_depth(child) for child in token.children)

    depths = []
    for sent in doc.sents:
        if sent.root:
            depths.append(get_depth(sent.root))
    return sum(depths) / len(depths) if depths else 0.0


def extract_english_features(text: str) -> Dict[str, Any]:
    """
    Extracts stylistic features from English text (with spaCy, NLTK, or Regex fallback).
    """
    if not text:
        return {}
        
    # Calculate baseline metrics on the FULL text using fast regex/string operations
    words = re.findall(r'\b\w+\b', text.lower())
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    dialogue_lines = [l for l in lines if l.startswith('"') or l.startswith("'") or l.startswith('“') or l.startswith('”')]
    punc_count = len(re.findall(r'[.,\/#!$%\^&\*;:{}=\-_`~()?"\']', text))
    
    word_count = len(words)
    sentence_count = len(sentences)
    avg_sentence_len = word_count / sentence_count if sentence_count > 0 else 0.0
    dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    punc_density = punc_count / len(text) if len(text) > 0 else 0.0

    # New features: avg_sentences_per_paragraph and compound_sentiment
    para_sentence_counts = [len([s.strip() for s in re.split(r'[.!?]+', p) if s.strip()]) for p in lines]
    avg_sentences_per_paragraph = sum(para_sentence_counts) / len(lines) if lines else 0.0

    compound_sentiment = 0.0
    if HAS_NLTK:
        try:
            import nltk
            from nltk.sentiment.vader import SentimentIntensityAnalyzer
            sia = SentimentIntensityAnalyzer()
            compound_sentiment = sia.polarity_scores(text)["compound"]
        except Exception:
            pass
    
    # Fallback/default metrics
    dep_tree_depth = 0.0
    adj_ratio = 0.0
    verb_ratio = 0.0
    pron_ratio = 0.0
    entity_density = 0.0
    ttr = compute_type_token_ratio(words)
    
    # Extract advanced features on a sample limit of 10,000 characters
    if HAS_SPACY and nlp_en is not None:
        try:
            sample_text = text[:10000]
            doc = nlp_en(sample_text)
            sample_words = [t for t in doc if not t.is_punct and not t.is_space]
            sample_word_count = len(sample_words)
            if sample_word_count > 0:
                lemmas = [t.lemma_.lower() for t in sample_words]
                ttr = compute_type_token_ratio(lemmas)
                dep_tree_depth = compute_dep_tree_depth(doc)
                
                adj_count = len([t for t in doc if t.pos_ == "ADJ"])
                verb_count = len([t for t in doc if t.pos_ in ("VERB", "AUX")])
                pron_count = len([t for t in doc if t.pos_ == "PRON"])
                
                adj_ratio = adj_count / sample_word_count
                verb_ratio = verb_count / sample_word_count
                pron_ratio = pron_count / sample_word_count
                
                entity_count = len(doc.ents)
                entity_density = (entity_count / sample_word_count) * 100
        except Exception as e:
            print(f"Error in English spaCy features extraction: {e}")
            
    elif HAS_NLTK:
        try:
            sample_text = text[:10000]
            sample_words = nltk.word_tokenize(sample_text.lower())
            sample_word_count = len(sample_words)
            if sample_word_count > 0:
                tagged = nltk.pos_tag(sample_words)
                adj_count = len([w for w, tag in tagged if tag in ('JJ', 'JJR', 'JJS')])
                verb_count = len([w for w, tag in tagged if tag.startswith('V') or tag == 'MD'])
                pron_count = len([w for w, tag in tagged if tag in ('PRP', 'PRP$', 'WP', 'WP$')])
                
                adj_ratio = adj_count / sample_word_count
                verb_ratio = verb_count / sample_word_count
                pron_ratio = pron_count / sample_word_count
        except Exception:
            pass

    # Extended Arxiv feature metrics (English)
    theme_words = ["love", "justice", "truth", "death", "fate", "honor", "humanity", "destiny", "wisdom", "morality", "grief", "joy", "peace", "war", "hope", "despair", "time", "memory", "soul", "mind", "life", "world", "history", "nature"]
    theme_pattern = r'\b(' + '|'.join(theme_words) + r')\b'
    theme_count = len(re.findall(theme_pattern, text.lower()))
    theme_explication_ratio = theme_count / max(1, word_count)

    linearity_words = ["remembered", "recalled", "flashback", "years ago", "decades ago", "months ago", "in the past", "formerly", "once", "suddenly", "memories", "yesterday", "tomorrow", "future", "past"]
    linearity_pattern = r'\b(' + '|'.join(linearity_words) + r')\b'
    linearity_count = len(re.findall(linearity_pattern, text.lower()))
    break_punc_count = len(re.findall(r'—|…|\.\.\.|\(|\)', text))
    linearity_subversion_score = (linearity_count + break_punc_count) / max(1, word_count)

    sensory_words = ["see", "hear", "smell", "taste", "feel", "touch", "look", "listen", "sound", "voice", "dark", "light", "red", "blue", "green", "black", "white", "cold", "hot", "warm", "sharp", "soft", "loud", "quiet", "eye", "hand", "face", "breath", "heart", "blood", "head", "body", "finger", "arm", "leg", "throat", "skin"]
    sensory_pattern = r'\b(' + '|'.join(sensory_words) + r')\b'
    sensory_count = len(re.findall(sensory_pattern, text.lower()))
    sensory_body_density = sensory_count / max(1, word_count)

    outside_words = ["sky", "wind", "rain", "sun", "moon", "star", "cloud", "street", "road", "building", "house", "city", "town", "tree", "forest", "mountain", "river", "sea", "ocean", "grass", "flower", "ground", "earth", "weather", "window", "door", "wall", "stone", "wood"]
    outside_pattern = r'\b(' + '|'.join(outside_words) + r')\b'
    outside_count = len(re.findall(outside_pattern, text.lower()))
    outside_world_engagement = outside_count / max(1, word_count)

    # Narrative feature diversity (English)
    vals = [ttr or 0.0, dialogue_ratio or 0.0, min(1.0, (punc_density or 0.0) * 10), verb_ratio or 0.0, adj_ratio or 0.0]
    mean_val = sum(vals) / len(vals)
    variance = sum((v - mean_val) ** 2 for v in vals) / len(vals)
    narrative_feature_diversity = float(1.0 / (1.0 + variance))
            
    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_sentence_len": avg_sentence_len,
        "dialogue_ratio": dialogue_ratio,
        "ttr": ttr,
        "punc_density": punc_density,
        "dep_tree_depth": dep_tree_depth,
        "adj_ratio": adj_ratio,
        "verb_ratio": verb_ratio,
        "pron_ratio": pron_ratio,
        "entity_density": entity_density,
        "avg_sentences_per_paragraph": avg_sentences_per_paragraph,
        "compound_sentiment": compound_sentiment,
        "theme_explication_ratio": theme_explication_ratio,
        "linearity_subversion_score": linearity_subversion_score,
        "sensory_body_density": sensory_body_density,
        "outside_world_engagement": outside_world_engagement,
        "narrative_feature_diversity": narrative_feature_diversity
    }

def extract_japanese_features(text: str) -> Dict[str, Any]:
    """
    Extracts stylistic features from Japanese text (with spaCy/Sudachi or Regex/NLTK fallback).
    """
    if not text:
        return {}
        
    # Calculate baseline metrics on the FULL text using fast string/regex operations
    chars = [c for c in text if not c.isspace()]
    sentences = [s.strip() for s in re.split(r'[。！？]+', text) if s.strip()]
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    dialogue_lines = [l for l in lines if l.startswith('「') or l.startswith('『')]
    punc_count = len(re.findall(r'[、。！？「」『』（）―…ー・]', text))
    
    char_count = len(chars)
    sentence_count = len(sentences)
    avg_sentence_len = char_count / sentence_count if sentence_count > 0 else 0.0
    dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    punc_density = punc_count / len(text) if len(text) > 0 else 0.0
    
    kanji_chars = re.findall(r'[\u4e00-\u9fff]', text)
    kanji_ratio = len(kanji_chars) / char_count if char_count > 0 else 0.0

    # New features: avg_sentences_per_paragraph and compound_sentiment
    para_sentence_counts = [len([s.strip() for s in re.split(r'[。！？]+', p) if s.strip()]) for p in lines]
    avg_sentences_per_paragraph = sum(para_sentence_counts) / len(lines) if lines else 0.0

    pos_words = ["嬉しい", "楽しい", "美しい", "素晴らしい", "愛する", "成功", "幸せ", "感謝", "満足"]
    neg_words = ["悲しい", "苦しい", "怒る", "嫌い", "失敗", "痛い", "最悪", "残念", "孤独"]
    pos_count = sum(text.count(w) for w in pos_words)
    neg_count = sum(text.count(w) for w in neg_words)
    compound_sentiment = (pos_count - neg_count) / (pos_count + neg_count + 1)
    
    # Defaults
    dep_tree_depth = 0.0
    particle_ratio = 0.0
    verb_ratio = 0.0
    ttr = compute_type_token_ratio(chars)
    
    # Advanced features on a sample limit of 10,000 characters
    if HAS_SPACY and nlp_ja is not None:
        try:
            sample_text = text[:10000]
            doc = nlp_ja(sample_text)
            sample_words = [t for t in doc if not t.is_punct and not t.is_space]
            sample_word_count = len(sample_words)
            if sample_word_count > 0:
                lemmas = [t.lemma_ for t in sample_words]
                ttr = compute_type_token_ratio(lemmas)
                dep_tree_depth = compute_dep_tree_depth(doc)
                
                particle_count = len([t for t in doc if t.pos_ == "ADP" or "助词" in t.tag_ or t.tag_.startswith("助詞")])
                verb_count = len([t for t in doc if t.pos_ in ("VERB", "AUX") or "动词" in t.tag_ or t.tag_.startswith("動詞")])
                
                particle_ratio = particle_count / sample_word_count
                verb_ratio = verb_count / sample_word_count
        except Exception as e:
            print(f"Error in Japanese spaCy features extraction: {e}")

    # Extended Arxiv feature metrics (Japanese)
    theme_words = ["愛", "正義", "真実", "死", "運命", "名誉", "人間", "宿命", "知恵", "道徳", "悲しみ", "喜び", "平和", "戦争", "希望", "絶望", "時間", "記憶", "魂", "心", "命", "世界", "歴史", "自然"]
    theme_count = sum(text.count(w) for w in theme_words)
    theme_explication_ratio = theme_count / max(1, char_count)

    linearity_words = ["思い出した", "回想", "昔", "過去", "以前", "かつて", "突然", "記憶", "昨日", "明日", "未来"]
    linearity_count = sum(text.count(w) for w in linearity_words)
    break_punc_count = len(re.findall(r'―|…|（|）|\(|\)', text))
    linearity_subversion_score = (linearity_count + break_punc_count) / max(1, char_count)

    sensory_words = ["見る", "聞く", "匂う", "味わう", "感じる", "触れる", "見る", "聴く", "音", "声", "暗い", "明るい", "赤い", "青い", "緑", "黒い", "白い", "冷たい", "熱い", "暖かい", "鋭い", "柔らかい", "うるさい", "静か", "目", "手", "顔", "息", "心臓", "血", "頭", "体", "指", "腕", "足", "喉", "肌"]
    sensory_count = sum(text.count(w) for w in sensory_words)
    sensory_body_density = sensory_count / max(1, char_count)

    outside_words = ["空", "風", "雨", "太陽", "月", "星", "雲", "通り", "道", "建物", "家", "都市", "町", "木", "森", "山", "川", "海", "芝生", "花", "地面", "地球", "天気", "窓", "ドア", "壁", "石", "木材"]
    outside_count = sum(text.count(w) for w in outside_words)
    outside_world_engagement = outside_count / max(1, char_count)

    # Narrative feature diversity (Japanese)
    vals = [ttr or 0.0, dialogue_ratio or 0.0, min(1.0, (punc_density or 0.0) * 5), verb_ratio or 0.0, particle_ratio or 0.0]
    mean_val = sum(vals) / len(vals)
    variance = sum((v - mean_val) ** 2 for v in vals) / len(vals)
    narrative_feature_diversity = float(1.0 / (1.0 + variance))
            
    return {
        "char_count": char_count,
        "sentence_count": sentence_count,
        "avg_sentence_len": avg_sentence_len,
        "dialogue_ratio": dialogue_ratio,
        "ttr": ttr,
        "punc_density": punc_density,
        "dep_tree_depth": dep_tree_depth,
        "particle_ratio": particle_ratio,
        "verb_ratio": verb_ratio,
        "kanji_ratio": kanji_ratio,
        "avg_sentences_per_paragraph": avg_sentences_per_paragraph,
        "compound_sentiment": compound_sentiment,
        "theme_explication_ratio": theme_explication_ratio,
        "linearity_subversion_score": linearity_subversion_score,
        "sensory_body_density": sensory_body_density,
        "outside_world_engagement": outside_world_engagement,
        "narrative_feature_diversity": narrative_feature_diversity
    }


def extract_chinese_features(text: str) -> Dict[str, Any]:
    """
    Extracts stylistic features from Chinese text (utilizing spaCy, NLTK, HanLP, or Regex fallback).
    """
    if not text:
        return {}
        
    # Calculate baseline metrics on the FULL text using fast string/regex operations
    chars = [c for c in text if not c.isspace()]
    char_count = len(chars)
    sentences = [s.strip() for s in re.split(r'[\u3002\uff01\uff1f\n]+', text) if s.strip()]
    sentence_count = len(sentences)
    avg_sentence_len = char_count / sentence_count if sentence_count > 0 else 0.0
    
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    dialogue_lines = [l for l in lines if l.startswith('\u201c') or l.startswith('\u300c') or l.startswith('\u5b89\u5168') or l.startswith('\u300e')]
    dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    
    ttr = compute_type_token_ratio(chars)

    # New features: avg_sentences_per_paragraph and compound_sentiment
    para_sentence_counts = [len([s.strip() for s in re.split(r'[。！？]+', p) if s.strip()]) for p in lines]
    avg_sentences_per_paragraph = sum(para_sentence_counts) / len(lines) if lines else 0.0

    pos_words = ["高兴", "开心", "美丽", "棒", "爱", "成功", "幸福", "感谢", "满意", "喜欢"]
    neg_words = ["悲伤", "痛苦", "生气", "讨厌", "失败", "疼", "差", "可惜", "孤独", "难过"]
    pos_count = sum(text.count(w) for w in pos_words)
    neg_count = sum(text.count(w) for w in neg_words)
    compound_sentiment = (pos_count - neg_count) / (pos_count + neg_count + 1)
    punc_count = len(re.findall(r'[\uff0c\u3001\u3002\uff01\uff1f\uff1b\uff1a""\u2018\u2019\uff08\uff09\u300a\u300b\u3010\u3011\u300e\u300f\u300c\u300d\u2014\u2014\u2026\u2026]', text))
    punc_density = punc_count / len(text) if len(text) > 0 else 0.0
    
    dep_tree_depth = 0.0
    particle_ratio = 0.0
    verb_ratio = 0.0
    
    # Advanced features on a sample limit of 10,000 characters
    success = False
    if HAS_HANLP and nlp_hanlp is not None:
        try:
            sample_text = text[:10000]
            doc = nlp_hanlp(sample_text)
            words = doc.get('tok') or doc.get('tok/fine') or doc.get('tok/coarse')
            if words:
                word_count = len(words)
                ttr = compute_type_token_ratio(words)
                
                pos_tags = doc.get('pos') or doc.get('pos/pku') or doc.get('pos/ctb') or doc.get('pos/863')
                if pos_tags:
                    particle_count = sum(1 for tag in pos_tags if tag in ('u', 'y', '\u52a9\u8bcd') or tag.startswith(('u', 'y')))
                    verb_count = sum(1 for tag in pos_tags if tag in ('v', 'vd', 'vn', '\u52a8\u8bcd') or tag.startswith('v'))
                    particle_ratio = particle_count / word_count if word_count > 0 else 0.0
                    verb_ratio = verb_count / word_count if word_count > 0 else 0.0
                    
                heads = doc.get('dep')
                if heads:
                    def get_node_depth(idx, memo):
                        if idx in memo:
                            return memo[idx]
                        parent = heads[idx][0]
                        if parent == 0:
                            memo[idx] = 1
                            return 1
                        val = 1 + get_node_depth(parent - 1, memo)
                        memo[idx] = val
                        return val
                    memo = {}
                    depths = [get_node_depth(i, memo) for i in range(len(heads))]
                    dep_tree_depth = max(depths) if depths else 0.0
            success = True
        except Exception:
            pass
            
    if not success and HAS_SPACY and nlp_zh is not None:
        try:
            sample_text = text[:10000]
            doc = nlp_zh(sample_text)
            words = [t for t in doc if not t.is_punct and not t.is_space]
            word_count = len(words)
            if word_count > 0:
                lemmas = [t.lemma_ for t in words]
                ttr = compute_type_token_ratio(lemmas)
                dep_tree_depth = compute_dep_tree_depth(doc)
                
                particle_count = len([t for t in doc if t.pos_ in ("ADP", "PART")])
                verb_count = len([t for t in doc if t.pos_ in ("VERB", "AUX")])
                
                particle_ratio = particle_count / word_count
                verb_ratio = verb_count / word_count
            success = True
        except Exception:
            pass

    # Extended Arxiv feature metrics (Chinese)
    theme_words = ["爱", "正义", "真实", "死", "命运", "名誉", "人类", "宿命", "智慧", "道德", "悲伤", "喜悦", "和平", "战争", "希望", "绝望", "时间", "记忆", "灵魂", "心", "生命", "世界", "历史", "自然"]
    theme_count = sum(text.count(w) for w in theme_words)
    theme_explication_ratio = theme_count / max(1, char_count)

    linearity_words = ["想起", "回忆", "以前", "过去", "曾经", "突然", "记忆", "昨天", "明天", "未来"]
    linearity_count = sum(text.count(w) for w in linearity_words)
    break_punc_count = len(re.findall(r'——|……|（|）|\(|\)', text))
    linearity_subversion_score = (linearity_count + break_punc_count) / max(1, char_count)

    sensory_words = ["看", "听", "闻", "尝", "感觉", "触摸", "瞧", "声音", "嗓音", "黑暗", "明亮", "红色", "蓝色", "绿色", "黑色", "白色", "冷", "热", "温暖", "锋利", "柔软", "吵闹", "安静", "眼睛", "手", "脸", "呼吸", "心脏", "血液", "头", "身体", "手指", "手臂", "腿", "喉咙", "皮肤"]
    sensory_count = sum(text.count(w) for w in sensory_words)
    sensory_body_density = sensory_count / max(1, char_count)

    outside_words = ["天空", "风", "雨", "太阳", "月亮", "星星", "云", "街道", "路", "建筑物", "房子", "城市", "城镇", "树", "森林", "山", "河流", "海", "草", "花", "地面", "地球", "天气", "窗户", "门", "墙", "石头", "木头"]
    outside_count = sum(text.count(w) for w in outside_words)
    outside_world_engagement = outside_count / max(1, char_count)

    # Narrative feature diversity (Chinese)
    vals = [ttr or 0.0, dialogue_ratio or 0.0, min(1.0, (punc_density or 0.0) * 5), verb_ratio or 0.0, particle_ratio or 0.0]
    mean_val = sum(vals) / len(vals)
    variance = sum((v - mean_val) ** 2 for v in vals) / len(vals)
    narrative_feature_diversity = float(1.0 / (1.0 + variance))
            
    return {
        "char_count": char_count,
        "sentence_count": sentence_count,
        "avg_sentence_len": avg_sentence_len,
        "dialogue_ratio": dialogue_ratio,
        "ttr": ttr,
        "punc_density": punc_density,
        "dep_tree_depth": dep_tree_depth,
        "particle_ratio": particle_ratio,
        "verb_ratio": verb_ratio,
        "avg_sentences_per_paragraph": avg_sentences_per_paragraph,
        "compound_sentiment": compound_sentiment,
        "theme_explication_ratio": theme_explication_ratio,
        "linearity_subversion_score": linearity_subversion_score,
        "sensory_body_density": sensory_body_density,
        "outside_world_engagement": outside_world_engagement,
        "narrative_feature_diversity": narrative_feature_diversity
    }

import math

MIN_MAX_BOUNDS = {
    "ttr": (0.01, 0.60),
    "dialogue_ratio": (0.0, 0.8),
    "punc_density": (0.0, 0.25),
    "dep_tree_depth": (0.0, 8.0),
    "verb_ratio": (0.0, 0.4),
    "avg_sentences_per_paragraph": (1.0, 10.0),
    "compound_sentiment": (-1.0, 1.0),
    "theme_explication_ratio": (0.0, 0.05),
    "linearity_subversion_score": (0.0, 0.05),
    "sensory_body_density": (0.0, 0.1),
    "outside_world_engagement": (0.0, 0.1),
    "narrative_feature_diversity": (0.0, 1.0)
}

ARCHETYPES = {
    "Victorian Novel": {
        "ttr": 0.8,
        "dialogue_ratio": 0.4,
        "punc_density": 0.5,
        "dep_tree_depth": 0.85,
        "verb_ratio": 0.45,
        "avg_sentences_per_paragraph": 0.6,
        "compound_sentiment": 0.5,
        "theme_explication_ratio": 0.6,
        "linearity_subversion_score": 0.3,
        "sensory_body_density": 0.7,
        "outside_world_engagement": 0.7,
        "narrative_feature_diversity": 0.8
    },
    "Philosophical Fiction": {
        "ttr": 0.85,
        "dialogue_ratio": 0.2,
        "punc_density": 0.4,
        "dep_tree_depth": 0.8,
        "verb_ratio": 0.5,
        "avg_sentences_per_paragraph": 0.7,
        "compound_sentiment": 0.4,
        "theme_explication_ratio": 0.95,
        "linearity_subversion_score": 0.4,
        "sensory_body_density": 0.3,
        "outside_world_engagement": 0.4,
        "narrative_feature_diversity": 0.8
    },
    "LitRPG": {
        "ttr": 0.3,
        "dialogue_ratio": 0.6,
        "punc_density": 0.7,
        "dep_tree_depth": 0.3,
        "verb_ratio": 0.6,
        "avg_sentences_per_paragraph": 0.2,
        "compound_sentiment": 0.5,
        "theme_explication_ratio": 0.2,
        "linearity_subversion_score": 0.9,
        "sensory_body_density": 0.6,
        "outside_world_engagement": 0.3,
        "narrative_feature_diversity": 0.4
    },
    "Isekai": {
        "ttr": 0.35,
        "dialogue_ratio": 0.65,
        "punc_density": 0.5,
        "dep_tree_depth": 0.35,
        "verb_ratio": 0.55,
        "avg_sentences_per_paragraph": 0.25,
        "compound_sentiment": 0.6,
        "theme_explication_ratio": 0.3,
        "linearity_subversion_score": 0.6,
        "sensory_body_density": 0.65,
        "outside_world_engagement": 0.4,
        "narrative_feature_diversity": 0.5
    },
    "Xianxia Cultivation": {
        "ttr": 0.4,
        "dialogue_ratio": 0.45,
        "punc_density": 0.4,
        "dep_tree_depth": 0.4,
        "verb_ratio": 0.5,
        "avg_sentences_per_paragraph": 0.3,
        "compound_sentiment": 0.4,
        "theme_explication_ratio": 0.75,
        "linearity_subversion_score": 0.5,
        "sensory_body_density": 0.8,
        "outside_world_engagement": 0.75,
        "narrative_feature_diversity": 0.6
    }
}

ARCHETYPE_TERRITORIES = {
    "Victorian Novel": "Classic Literature Territory",
    "Philosophical Fiction": "Classic Literature Territory",
    "LitRPG": "Web Novel Territory",
    "Isekai": "Web Novel Territory",
    "Xianxia Cultivation": "Web Novel Territory"
}

def match_archetype(features: dict) -> dict:
    prefix = ""
    for k in features.keys():
        if k.startswith("en_"):
            prefix = "en_"
            break
        elif k.startswith("ja_"):
            prefix = "ja_"
            break
        elif k.startswith("zh_"):
            prefix = "zh_"
            break
            
    agnostic = {}
    for k, v in features.items():
        if prefix and k.startswith(prefix):
            agnostic[k[len(prefix):]] = v
        elif not k.startswith("en_") and not k.startswith("ja_") and not k.startswith("zh_"):
            agnostic[k] = v
            
    # min-max normalization
    normalized = {}
    for key, bounds in MIN_MAX_BOUNDS.items():
        val = agnostic.get(key, None)
        if val is None:
            val = 0.0
        min_v, max_v = bounds
        norm_val = (val - min_v) / max(1e-9, max_v - min_v)
        norm_val = max(0.0, min(1.0, norm_val))
        normalized[key] = norm_val

    # Cosine similarity
    similarities = {}
    keys = list(MIN_MAX_BOUNDS.keys())
    input_norm = math.sqrt(sum(normalized[k] ** 2 for k in keys))
    
    best_trope = None
    best_sim = -1.0
    
    for trope, ref_vector in ARCHETYPES.items():
        dot_product = sum(normalized[k] * ref_vector[k] for k in keys)
        ref_norm = math.sqrt(sum(ref_vector[k] ** 2 for k in keys))
        
        if input_norm == 0.0 or ref_norm == 0.0:
            sim = 0.0
        else:
            sim = dot_product / (input_norm * ref_norm)
            
        similarities[trope] = sim
        if sim > best_sim:
            best_sim = sim
            best_trope = trope
            
    territory = ARCHETYPE_TERRITORIES.get(best_trope, "Unknown Territory")
    
    return {
        "territory": territory,
        "closest_trope": best_trope,
        "confidence": best_sim,
        "similarities": similarities
    }



In [5]:
# Load Database and Process Chapters

db_path = "data/kisholens.db" if os.path.exists("data/kisholens.db") else "../data/kisholens.db"
engine = create_engine(f"sqlite:///{db_path}")

features_list = []

with Session(engine) as session:
    novels = session.exec(select(Novel)).all()
    novels_map = {n.id: n for n in novels}
    
    chapters = session.exec(select(Chapter)).all()
    print(f"Processing {len(chapters)} chapters...")
    
    for ch in chapters:
        novel = novels_map.get(ch.novel_id)
        if not novel:
            continue
            
        row = {
            "novel_title": novel.title,
            "author": novel.author,
            "source": novel.source,
            "chapter_num": ch.chapter_number,
            "chapter_title": ch.title
        }
        
        if ch.text_en:
            en_feat = extract_english_features(ch.text_en)
            row.update({f"en_{k}": v for k, v in en_feat.items()})
            
        if ch.text_ja:
            ja_feat = extract_japanese_features(ch.text_ja)
            row.update({f"ja_{k}": v for k, v in ja_feat.items()})

        if hasattr(ch, 'text_zh') and ch.text_zh:
            zh_feat = extract_chinese_features(ch.text_zh)
            row.update({f"zh_{k}": v for k, v in zh_feat.items()})
            
        features_list.append(row)

df = pd.DataFrame(features_list)
df.head()

Processing 22 chapters...


,novel_title,author,source,chapter_num,chapter_title,en_word_count,en_sentence_count,en_avg_sentence_len,en_dialogue_ratio,en_ttr,...,zh_sentence_count,zh_avg_sentence_len,zh_dialogue_ratio,zh_ttr,zh_punc_density,zh_dep_tree_depth,zh_particle_ratio,zh_verb_ratio,zh_avg_sentences_per_paragraph,zh_compound_sentiment
0,Noble Reincarnation~Blessed With the Strongest...,三木なずな,syosetu,77,An Amateur's Perception,1454.0,142.0,10.239437,0.385246,0.296322,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Noble Reincarnation~Blessed With the Strongest...,三木なずな,syosetu,36,Aspiring Knight and the Knight,1710.0,182.0,9.395604,0.426573,0.274269,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sir Andy,mrsimple,scribblehub,1,Sir Andy,2768.0,219.0,12.639269,0.109091,0.336649,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,One Night as the Queen,mrsimple,scribblehub,1,One Night as the Queen,4115.0,324.0,12.700617,0.021739,0.330390,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Threadbare Titans,Pythonogram#,royalroad,1,Smoke and Silence,1364.0,169.0,8.071006,0.197183,0.436858,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Aggregate Stylistic Stats by Novel Source

print("Style Aggregates by Source Platform:")
group_cols = ["source"]
num_cols = [c for c in df.columns if c.startswith("en_") or c.startswith("ja_") or c.startswith("zh_")]
agg_df = df.groupby(group_cols)[num_cols].mean()
agg_df

Style Aggregates by Source Platform:


,en_word_count,en_sentence_count,en_avg_sentence_len,en_dialogue_ratio,en_ttr,en_punc_density,en_dep_tree_depth,en_adj_ratio,en_verb_ratio,en_pron_ratio,...,zh_sentence_count,zh_avg_sentence_len,zh_dialogue_ratio,zh_ttr,zh_punc_density,zh_dep_tree_depth,zh_particle_ratio,zh_verb_ratio,zh_avg_sentences_per_paragraph,zh_compound_sentiment
source,,,,,,,,,,,,,,,,,,,,,
cnnovel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3003.5,22.070428,0.294377,0.031155,0.092954,0.0,0.0,0.0,1.739561,0.083951
gutenberg,893.0,78.0,11.448718,0.348837,0.338964,0.036528,5.084746,0.072072,0.219595,0.194820,...,44038.0,13.479200,0.014729,0.007042,0.167554,0.0,0.0,0.0,2.504436,-0.653521
royalroad,992.5,113.0,9.482871,0.289068,0.426158,0.036730,4.298856,0.068833,0.213192,0.123412,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
scribblehub,3441.5,271.5,12.669943,0.065415,0.333519,0.033391,4.933231,0.083202,0.212774,0.163037,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
syosetu,1582.0,162.0,9.817521,0.405910,0.285296,0.062191,4.749657,0.051989,0.230313,0.178796,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Stylistic pacing comparisons (English texts)

en_cols = [c for c in df.columns if c.startswith("en_")]
if en_cols:
    print("English Style Aggregates by Novel:")
    novel_en = df.groupby(["novel_title", "source"])[en_cols].mean().dropna(how='all')
    print(novel_en.to_string())
else:
    print("No English style columns available.")

English Style Aggregates by Novel:
                                                                             en_word_count  en_sentence_count  en_avg_sentence_len  en_dialogue_ratio    en_ttr  en_punc_density  en_dep_tree_depth  en_adj_ratio  en_verb_ratio  en_pron_ratio  en_entity_density  en_avg_sentences_per_paragraph  en_compound_sentiment
novel_title                                                     source                                                                                                                                                                                                                                                   
Noble Reincarnation~Blessed With the Strongest Power From Birth syosetu             1582.0              162.0             9.817521           0.405910  0.285296         0.062191           4.749657      0.051989       0.230313       0.178796           4.142713                        1.659922                0.80975
One Night as the Queen 

In [8]:
# Stylistic pacing comparisons (Japanese texts)

ja_cols = [c for c in df.columns if c.startswith("ja_")]
if ja_cols:
    print("\nJapanese Style Aggregates by Novel:")
    novel_ja = df.groupby(["novel_title", "source"])[ja_cols].mean().dropna(how='all')
    print(novel_ja.to_string())
else:
    print("\nNo Japanese style columns available.")


Japanese Style Aggregates by Novel:
                                                                         ja_char_count  ja_sentence_count  ja_avg_sentence_len  ja_dialogue_ratio    ja_ttr  ja_punc_density  ja_dep_tree_depth  ja_particle_ratio  ja_verb_ratio  ja_kanji_ratio  ja_avg_sentences_per_paragraph  ja_compound_sentiment
novel_title                                                     source                                                                                                                                                                                                                                  
Noble Reincarnation~Blessed With the Strongest Power From Birth syosetu         3309.5               93.0             35.65795           0.402413  0.288198          0.10348           4.514803           0.316316       0.332034        0.248064                        1.134902                  0.125


In [9]:
# Stylistic pacing comparisons (Chinese texts)

zh_cols = [c for c in df.columns if c.startswith("zh_")]
if zh_cols:
    print("\nChinese Style Aggregates by Novel:")
    novel_zh = df.groupby(["novel_title", "source"])[zh_cols].mean().dropna(how='all')
    print(novel_zh.to_string())
else:
    print("\nNo Chinese style columns available.")


Chinese Style Aggregates by Novel:
                       zh_char_count  zh_sentence_count  zh_avg_sentence_len  zh_dialogue_ratio    zh_ttr  zh_punc_density  zh_dep_tree_depth  zh_particle_ratio  zh_verb_ratio  zh_avg_sentences_per_paragraph  zh_compound_sentiment
novel_title source                                                                                                                                                                                                                    
三國志演義       gutenberg       593597.0            44038.0            13.479200           0.014729  0.007042         0.167554                0.0                0.0            0.0                        2.504436              -0.653521
为爱入局：嫁给秦先生  cnnovel          57429.0             2624.0            21.886052           0.308129  0.036915         0.092420                0.0                0.0            0.0                        1.653434              -0.043210
绝世弃婿        cnnovel          75288.0    